In [2]:
import sys
!{sys.executable} -m pip install transformers

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
    --------------------------------------- 0.3/11.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.6 MB 1.2 MB/s eta 0:00:10
   -- ------------------------------------- 0.8/11.6 MB 1.0 MB/s eta 0:00:11
   --- ------------------------------------ 1.0/11.6 MB 1.0 MB/s eta 0:00:11
   --- ------------------------------------ 1.0/11.6 MB 1.0 MB/s eta 0:00:11
   ---- ----------------------------------- 1.3/11.6 MB 1.0 MB/s eta 0:00:11
   ----- ---------------------------------- 1.6/11.6 MB 1.0 MB/s eta 0:00:10
   ------ --------------------------------- 1.8/11.6 MB 1.0 MB/s eta 0:00:10
   ------- -------------------------------- 2.1/11.6 MB 1.0 MB/s eta 0:00:10
   -------- ------------------------------- 2.4/11.6 MB 1.0 MB/s eta 0:00:09
   --------- ------------------------------ 2.6/11.6 MB 1.1 MB/s eta 0:00:09
   --------- -------


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import torch
from transformers import BertTokenizerFast, BertModel
import numpy as np
from tqdm import tqdm
import pandas as pd

# Load CSV with URLs
csv_file = "Phishing_URL_Dataset.csv"
data = pd.read_csv(csv_file)
urls = data['url'].astype(str).tolist()

# Load tokenizer and base model
model_name = "CrabInHoney/urlbert-tiny-base-v4"
tokenizer = BertTokenizerFast.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Function to generate URL embeddings using base model
def generate_url_embeddings_verbose(url_list, batch_size=32):
    embeddings = []
    total_batches = (len(url_list) + batch_size - 1) // batch_size
    with torch.no_grad():
        for i in range(0, len(url_list), batch_size):
            batch_index = i // batch_size + 1
            batch_urls = url_list[i:i+batch_size]
            print(f"Processing batch {batch_index}/{total_batches} (URLs {i} to {i+len(batch_urls)-1})")

            encoded = tokenizer(batch_urls, padding=True, truncation=True, return_tensors='pt')
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

            print(f"Batch {batch_index} done, embeddings shape: {cls_embeddings.shape}")
            time.sleep(0.1)  # slight pause so prints/readable in console

    print("All batches processed, stacking embeddings")
    return np.vstack(embeddings)

# Generate URL embeddings
url_embeddings = generate_url_embeddings(urls)

# Assume engineered_features is precomputed numpy array with shape (num_samples, engineered_feature_dim)
# Example: engineered_features = np.load("engineered_features.npy")

# Concatenate URL embeddings and engineered features
#combined_features = np.hstack([engineered_features, url_embeddings])

# Now combined_features can be used as input for your TabM or other model training

print("URL embeddings shape:", url_embeddings.shape)
#print("Combined feature shape:", combined_features.shape)


Some weights of BertModel were not initialized from the model checkpoint at CrabInHoney/urlbert-tiny-base-v4 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|████████████████████████████████████████████████████████████████████████████████| 358/358 [00:49<00:00,  7.19it/s]

URL embeddings shape: (11430, 192)


In [7]:
print(url_embeddings)

[[ 0.64922297 -0.09423506 -0.11394003 ...  0.36659613  5.079102
   0.61710095]
 [ 1.5780351   1.1581514   1.4655658  ... -0.55513084  4.2040925
  -0.13807054]
 [ 1.8369963   1.2684455   2.169752   ...  0.1830782   2.0814097
  -0.6586529 ]
 ...
 [ 0.886112    1.5763144   1.5370874  ... -0.32667857  4.101621
   0.695455  ]
 [ 0.62165844  0.11378357  0.04939958 ...  0.29608527  5.05585
   0.38443887]
 [ 3.05993    -0.9369401  -0.7134068  ... -2.8007963   1.2162383
  -0.4703322 ]]


In [8]:
import pandas as pd
import numpy as np

# Load original CSV
csv_file = "Phishing_URL_Dataset.csv"
df = pd.read_csv(csv_file)

# url_embeddings: numpy array of shape (num_samples, embedding_dim)
# Current ofrmat of Embeddings: url_embeddings = np.array([[...], [...], ...])

# Create DataFrame from embeddings, name columns embedding_0, embedding_1, ...
embedding_cols = [f"embedding_{i}" for i in range(url_embeddings.shape[1])]
df_embeddings = pd.DataFrame(url_embeddings, columns=embedding_cols)

# Insert embeddings after the url column (assumed index 1)
url_col_pos = df.columns.get_loc('url') + 1
df = pd.concat([df.iloc[:, :url_col_pos], df_embeddings, df.iloc[:, url_col_pos:]], axis=1)

# Save modified CSV with embeddings
df.to_csv("Phishing_URL_Dataset_with_embeddings.csv", index=False)

print(f"Inserted {url_embeddings.shape[1]} embedding columns after 'url' column")
print(df.head())


Inserted 192 embedding columns after 'url' column
                                                 url  embedding_0  \
0              http://www.crestonwood.com/router.php     0.649223   
1  http://shadetreetechnology.com/V4/validation/a...     1.578035   
2  https://support-appleld.com.secureupdate.duila...     1.836996   
3                                 http://rgipt.ac.in     0.269069   
4  http://www.iracing.com/tracks/gateway-motorspo...     0.391744   

   embedding_1  embedding_2  embedding_3  embedding_4  embedding_5  \
0    -0.094235    -0.113940     0.130148     0.377326     0.606072   
1     1.158151     1.465566    -0.156674     0.838531     1.205377   
2     1.268445     2.169752    -1.207129     2.030680     3.250151   
3    -0.472855    -0.641673     0.384676     0.071700     0.994376   
4     0.402081     0.148963    -0.073358     0.681523     0.378718   

   embedding_6  embedding_7  embedding_8  ...  domain_in_title  \
0     1.711852     0.286139    -0.079585  ...   